# Lab 11: Sieci konwolucyjne — własna architektura i modele pretrenowane
### Biblioteki Python w analizie danych
**Tomasz Rodak**

[![](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rodakt/BPwAD/blob/v2/laby/lab_11.ipynb)

Wykład 6 wprowadził sieci konwolucyjne — kanoniczne klocki (`nn.Conv2d`, `nn.MaxPool2d`, `nn.BatchNorm2d`), wzory na rozmiar wyjścia i liczbę parametrów, oraz klasyczną architekturę LeNet-5 na zbiorze MNIST. W tym laboratorium wykonamy trzy kroki, które utrwalą ten materiał i pierwszy raz zetkną nas z modelami pretrenowanymi:

1. **Sekcja 1**: rozgrzewka rachunkowa — rozmiary tensorów, liczba parametrów per warstwa, receptive field. Liczymy na papierze, weryfikujemy w PyTorch.
2. **Sekcja 2**: budujemy od zera nietrywialną CNN dla zbioru CIFAR-10 (kolorowe obrazy 32 × 32, 10 klas), z trzema blokami konwolucyjnymi, BatchNorm i Dropoutem, i trenujemy ją do ~70% accuracy testowej. To pierwszy lab kursu, w którym trening na CPU byłby niepraktyczny — włącz GPU.
3. **Sekcje 3–4**: otwieramy "pod maską" pretrenowane modele z `torchvision.models` (ResNet-18, ResNet-34, ResNet-50, VGG16). Liczymy ich parametry, wizualizujemy filtry pierwszej warstwy ResNet-18, i potwierdzamy, że model w stanie surowym klasyfikuje obrazy z ImageNet.

Pretrenowane modele wracają w wykładzie 7 i lab 12 jako narzędzie *transfer learningu* — najpierw musimy je jednak zobaczyć od środka.

**GPU w Colab.** Przed uruchomieniem przejdź do *Środowisko wykonawcze → Zmień typ środowiska* i wybierz akcelerator GPU (typowo T4). Sekcja 2 jest zaprojektowana tak, żeby mieścić się w darmowym limicie czasu (~5 min treningu).

In [ ]:
import io
import requests
from PIL import Image

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader

import torchvision
import torchvision.transforms as transforms
from torchvision import datasets
from torchvision.models import (
    resnet18, ResNet18_Weights,
    resnet34, resnet50, vgg16,
)

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay


In [ ]:
#| eval: false
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Urządzenie: {device}")


## 1. Rozgrzewka: rozmiary i parametry warstw konwolucyjnych

Dwa krótkie zadania rachunkowe z wzorów wprowadzonych na wykładzie 6. Wykonaj je **na papierze** (lub w komentarzach), a wynik zweryfikuj w PyTorch — zbuduj sieć, przepuść przez nią dummy input i porównaj kształty oraz sumę parametrów z własnymi obliczeniami.

Przypomnienie wzorów:

- **Rozmiar wyjścia warstwy konwolucyjnej (jeden wymiar przestrzenny)**: dla wejścia o rozmiarze $W$, filtra $K \times K$, paddingu $P$ i stride $S$:

  $$W_{\text{out}} = \left\lfloor \frac{W - K + 2P}{S} \right\rfloor + 1.$$

  Dla `MaxPool2d(kernel_size=K, stride=S)` wzór jest identyczny (domyślnie $P = 0$).
- **Liczba parametrów warstwy `Conv2d(C_in, C_out, K)`** (z biasami, domyślnie `bias=True`):

  $$C_{\text{out}} \cdot \bigl(C_{\text{in}} \cdot K^2 + 1\bigr).$$

  Warstwy `MaxPool2d` nie mają parametrów.

### 1.1 Dwie konwolucje, bez poolingu

Rozważ sieć:

```
Wejście: 3 × 64 × 64 (RGB)
Conv2d(in_channels=3,  out_channels=16, kernel_size=3, stride=1, padding=1)
Conv2d(in_channels=16, out_channels=32, kernel_size=3, stride=1, padding=0)
```

Oblicz:

1. Rozmiar wyjścia po **każdej** z dwóch warstw (jako trójka kanały × wysokość × szerokość).
2. Liczbę parametrów **każdej** warstwy z osobna oraz łącznie.

Zbuduj sieć jako `nn.Sequential`, przepuść przez nią `torch.randn(1, 3, 64, 64)` i zweryfikuj kształt wyjścia. Sumę parametrów policz przez

```python
sum(p.numel() for p in model.parameters())
```

Czy padding w drugiej warstwie został dobrany tak, żeby zachować rozdzielczość przestrzenną? Jak zmieniłby się rozmiar wyjścia, gdybyśmy ustawili `padding=1`?

### 1.2 Konwolucje z poolingiem, receptive field

Bardziej realistyczna konfiguracja:

```
Wejście: 1 × 224 × 224 (np. duży obraz w skali szarości)
Conv2d(1,  16, kernel_size=5, stride=1, padding=0)
MaxPool2d(kernel_size=2, stride=2)
Conv2d(16, 32, kernel_size=3, stride=1, padding=0)
MaxPool2d(kernel_size=2, stride=2)
```

Oblicz:

1. Rozmiar wyjścia po **każdej** z czterech warstw.
2. Liczbę parametrów każdej warstwy z osobna oraz łącznie.
3. **Receptive field**: jaki obszar obrazu wejściowego "widzi" pojedynczy neuron w mapie cech *po* ostatnim MaxPool? A pojedynczy neuron *po* pierwszym Conv?

*Wzór na receptive field.* Niech warstwa ma filtr o rozmiarze $k$ i stride $s$, a poprzednio nagromadzony "skok" (iloczyn stride'ów wszystkich wcześniejszych warstw) wynosi $j_{\text{prev}}$, a receptive field — $r_{\text{prev}}$. Wtedy po tej warstwie:

$$r_{\text{new}} = r_{\text{prev}} + (k - 1) \cdot j_{\text{prev}}, \qquad j_{\text{new}} = j_{\text{prev}} \cdot s.$$

Startujemy z $r_0 = 1$, $j_0 = 1$ (pojedynczy piksel "widzi" sam siebie, skok między sąsiednimi neuronami wejściowymi to 1 piksel).

Zbuduj sieć w PyTorch i zweryfikuj kształty wyjść.

**Intuicja: dlaczego receptive field ma znaczenie.** Sieć konwolucyjna może rozpoznawać tylko te wzorce, które mieszczą się w receptive field jej ostatnich warstw. Każdy pooling podwaja "skok" $j$, a kolejne konwolucje sięgają w coraz większy obszar oryginalnego obrazu — głębokie sieci typu VGG na obrazach 224 × 224 osiągają RF rzędu pełnego obrazu. To dlatego dla problemów wymagających kontekstu (segmentacja semantyczna, klasyfikacja sceny) potrzebujemy głębokich sieci, a nie tylko wielu filtrów w jednej warstwie.

## 2. Własna CNN na CIFAR-10

CIFAR-10 to klasyczny zbiór 60 000 obrazów RGB o rozmiarze 32 × 32, podzielony na 10 klas: samolot, samochód, ptak, kot, jeleń, pies, żaba, koń, statek, ciężarówka. 50 000 obrazów stanowi zbiór treningowy, 10 000 — testowy. Mimo skromnej rozdzielczości jest to zauważalnie trudniejszy problem niż MNIST: tła są naturalne, kolory pełnoskalowe, klasy mają dużą zmienność wewnętrzną (kot na drzewie i kot w salonie to dwie zupełnie różne tekstury).

Naszym celem jest osiągnięcie ~70% accuracy testowej w 15 epokach treningu od zera. To dobry wynik dla CNN bez transfer learningu — pretrenowane modele osiągają na CIFAR-10 powyżej 95% accuracy, do czego wrócimy w lab 12.

### 2.1 Dane

`torchvision.datasets.CIFAR10` pobiera zbiór automatycznie i opakowuje go w obiekt zwracający pary `(obraz, etykieta)`. Statystyki normalizacji (średnia i odchylenie standardowe per kanał) są dobrze znane:

In [ ]:
#| eval: false
CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD  = (0.2470, 0.2435, 0.2616)


Transformacje:

- **Trening:** `RandomCrop(32, padding=4)` (losowe wycięcie 32 × 32 z obrazu dopaddingowanego do 40 × 40 zerami — najprostszy augmentation), `RandomHorizontalFlip()` (lustrzane odbicie z prawdopodobieństwem 0.5), `ToTensor()`, `Normalize(...)`.
- **Test:** wyłącznie `ToTensor()` i `Normalize(...)` z **tymi samymi** statystykami.

In [ ]:
#| eval: false
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD),
])
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD),
])

train_set = datasets.CIFAR10(root="./data", train=True,  download=True, transform=transform_train)
test_set  = datasets.CIFAR10(root="./data", train=False, download=True, transform=transform_test)

train_loader = DataLoader(train_set, batch_size=128, shuffle=True,  num_workers=2)
test_loader  = DataLoader(test_set,  batch_size=256, shuffle=False, num_workers=2)

CLASSES = ("plane", "car", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck")
print(f"Train: {len(train_set)} obrazów, Test: {len(test_set)}")


**Dlaczego augmentation, a nie więcej epok.** Augmentation (`RandomCrop`, `RandomHorizontalFlip`) sztucznie zwiększa zmienność zbioru treningowego — model w każdej epoce widzi lekko inne wersje tych samych obrazów. To redukuje overfitting znacznie skuteczniej niż samo wydłużenie treningu, którego efektem byłoby raczej zapamiętanie zbioru treningowego niż lepsza generalizacja. Tę technikę zobaczymy ponownie w lab 12 w wydaniu fastai.

**Higiena.** Na zbiorze testowym **nie używamy** augmentacji ani shuffle — predykcje muszą być deterministyczne i porównywalne między uruchomieniami. Augmentation ma sens tylko w treningu, gdzie model "płaci" za swoje błędy gradientem.

Opcjonalnie: podejrzyj jeden batch z `train_loader` i narysuj 8 obrazów w siatce z etykietami. Pamiętaj o **denormalizacji** przed `imshow` (wzór dalej w sekcji 2.5).

### 2.2 Architektura

Zbudujemy CNN o trzech blokach konwolucyjnych. Każdy blok to dwie warstwy konwolucyjne z BatchNorm i ReLU, zakończone MaxPool. Po blokach następuje klasyfikator MLP z dropoutem.

```
Wejście: 3 × 32 × 32

Block 1: Conv(3→32,   3×3, p=1) → BN → ReLU
         Conv(32→32,  3×3, p=1) → BN → ReLU
         MaxPool(2×2)                                 → 32 × 16 × 16

Block 2: Conv(32→64,  3×3, p=1) → BN → ReLU
         Conv(64→64,  3×3, p=1) → BN → ReLU
         MaxPool(2×2)                                 → 64 × 8 × 8

Block 3: Conv(64→128, 3×3, p=1) → BN → ReLU
         Conv(128→128, 3×3, p=1) → BN → ReLU
         MaxPool(2×2)                                 → 128 × 4 × 4

Klasyfikator:
         Flatten                                      → 2048
         Dropout(0.5) → Linear(2048→256) → ReLU
         Dropout(0.5) → Linear(256→10)                → 10 (logity)
```

**Krótko o nowych klockach.**

- **`nn.BatchNorm2d(C)`** normalizuje aktywacje w trakcie treningu na poziomie batcha (per kanał: odejmuje średnią batcha, dzieli przez odchylenie standardowe batcha, potem skaluje i przesuwa przez wyuczone parametry $\gamma, \beta$). W praktyce stabilizuje trening i pozwala na większe `lr`. Ma dwa parametry uczone na kanał — to widać w sumie parametrów.
- **`nn.Dropout(p)`** w trakcie treningu losowo zeruje frakcję $p$ wejść, co utrudnia modelowi nadmierne uzależnienie się od konkretnych neuronów. W trybie `model.eval()` Dropout jest wyłączany; BatchNorm również przechodzi w "tryb inferencyjny" i korzysta ze średnich ruchomych zapamiętanych z treningu, a nie ze statystyk bieżącego batcha. To jest powód, dla którego `model.train()` / `model.eval()` musi być wywoływane w odpowiednich miejscach pętli.

**Ćwiczenie do samodzielnego przeliczenia (przed kodem).** Policz **liczbę parametrów każdej warstwy** oraz **sumę**. Zwróć uwagę, że BatchNorm ma 2 parametry uczone na kanał (są jeszcze średnie ruchome, ale te nie są parametrami uczonymi gradientem — `running_mean` i `running_var` nie są zwracane przez `model.parameters()`). Po napisaniu modelu zweryfikuj swój wynik:

```python
sum(p.numel() for p in model.parameters())
```

Spodziewaj się liczby w przedziale kilkuset tysięcy parametrów. Większość będzie zlokalizowana w pierwszej warstwie liniowej klasyfikatora — sprawdź, którą cząstkę całości stanowi `Linear(2048, 256)`.

Zaproponowany szkielet (uzupełnij brakujące warstwy):

In [ ]:
#| eval: false
class CIFAR10CNN(nn.Module):
    def __init__(self, num_classes=10, dropout=0.5):
        super().__init__()
        # --- Block 1 ---
        self.conv1a = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.bn1a   = nn.BatchNorm2d(32)
        self.conv1b = ...    # uzupełnij
        self.bn1b   = ...
        # --- Block 2 ---
        ...
        # --- Block 3 ---
        ...
        # MaxPool można współdzielić — nie ma parametrów uczonych
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        # --- Klasyfikator ---
        self.dropout = nn.Dropout(dropout)
        self.fc1 = nn.Linear(128 * 4 * 4, 256)
        self.fc2 = nn.Linear(256, num_classes)

    def forward(self, x):
        # Block 1
        x = F.relu(self.bn1a(self.conv1a(x)))
        x = F.relu(self.bn1b(self.conv1b(x)))
        x = self.pool(x)
        # Block 2
        ...
        # Block 3
        ...
        # Klasyfikator
        x = torch.flatten(x, start_dim=1)   # (N, 128*4*4)
        x = self.dropout(x)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x                            # logity, bez softmaxa


Sieć zwraca logity — funkcja straty `nn.CrossEntropyLoss` zawiera softmax w środku (porównaj z lab 9 sekcja 5.3).

Po napisaniu modelu wykonaj sanity check kształtów i liczby parametrów:

In [ ]:
#| eval: false
model = CIFAR10CNN().to(device)
dummy = torch.randn(2, 3, 32, 32, device=device)
out = model(dummy)
print(out.shape)                                                    # oczekiwane: torch.Size([2, 10])
print(f"Parametry: {sum(p.numel() for p in model.parameters()):,}") # porównaj z własnym rachunkiem


### 2.3 Pętla treningowa

Kanoniczna pętla z lab 9–10, rozszerzona o przeniesienie batchy na `device`. Funkcja straty: `nn.CrossEntropyLoss`. Optymalizator: `optim.Adam(model.parameters(), lr=1e-3)`. Liczba epok: 15.

W trakcie treningu śledź trzy wielkości i zapisuj do list:

- **`train_loss_history`** — średnia strata na zbiorze treningowym (uśredniona po batchach w epoce),
- **`test_loss_history`** — średnia strata na zbiorze testowym po epoce,
- **`test_acc_history`** — accuracy na zbiorze testowym po epoce.

Ewaluacja na zbiorze testowym po każdej epoce — w `model.eval()` + `with torch.no_grad()`. Pamiętaj o `model.train()` przed kolejną epoką treningu (BatchNorm i Dropout zachowują się różnie w obu trybach).

Szkielet pętli (uzupełnij ciało):

In [ ]:
#| eval: false
model     = CIFAR10CNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

train_loss_history = []
test_loss_history  = []
test_acc_history   = []

NUM_EPOCHS = 15

for epoch in range(NUM_EPOCHS):
    # --- TRAIN ---
    model.train()
    running_loss = 0.0
    n_seen = 0
    for X, y in train_loader:
        X, y = X.to(device), y.to(device)
        # zero_grad → forward → loss → backward → step
        ...
        running_loss += loss.item() * X.size(0)
        n_seen += X.size(0)
    train_loss = running_loss / n_seen

    # --- EVAL ---
    model.eval()
    test_loss = 0.0
    correct   = 0
    n_seen    = 0
    with torch.no_grad():
        for X, y in test_loader:
            X, y = X.to(device), y.to(device)
            logits = model(X)
            loss   = criterion(logits, y)
            test_loss += loss.item() * X.size(0)
            correct   += (logits.argmax(dim=1) == y).sum().item()
            n_seen    += X.size(0)
    test_loss /= n_seen
    test_acc   = correct / n_seen

    train_loss_history.append(train_loss)
    test_loss_history.append(test_loss)
    test_acc_history.append(test_acc)

    print(f"Epoka {epoch+1:2d}/{NUM_EPOCHS}  "
          f"train loss={train_loss:.4f}  test loss={test_loss:.4f}  test acc={test_acc:.4f}")


*Wskazówka czasowa.* Na GPU T4 w Colab jedna epoka zajmuje ~20–30 sekund, cały trening to ~5 minut. Na CPU byłoby to ponad godzinę — upewnij się, że `device` to `cuda`.

### 2.4 Krzywe uczenia i ocena

Narysuj dwa wykresy obok siebie (`fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))`):

1. **Strata** treningowa i testowa w funkcji epoki, na jednej osi.
2. **Accuracy** testowa w funkcji epoki.

Końcowa accuracy powinna mieścić się w okolicach 70%. Jeśli osiągniesz wyraźnie mniej, sprawdź:

- czy `Normalize` jest aktywne w **obu** transformacjach,
- czy `.to(device)` zostało wywołane na modelu **i** na każdym batchu w obu pętlach (train i eval),
- czy wymiar wejściowy `fc1` to faktycznie $128 \cdot 4 \cdot 4 = 2048$,
- czy nie zapomniałeś `model.train()` na początku epoki (Dropout pozostawiony w trybie inferencyjnym = brak regularyzacji).

Policz i narysuj **macierz pomyłek** na zbiorze testowym:

In [ ]:
#| eval: false
model.eval()
all_preds, all_targets = [], []
with torch.no_grad():
    for X, y in test_loader:
        logits = model(X.to(device))
        all_preds.append(logits.argmax(dim=1).cpu().numpy())
        all_targets.append(y.numpy())
y_pred = np.concatenate(all_preds)
y_true = np.concatenate(all_targets)

cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(8, 7))
ConfusionMatrixDisplay(cm, display_labels=CLASSES).plot(ax=ax, xticks_rotation=45, colorbar=False)
ax.set_title("Macierz pomyłek — CIFAR-10")


**Co zauważyć w macierzy.** Pary klas mylące się są charakterystyczne i powtarzalne między różnymi CNN-ami: **kot ↔ pies** (oba puchate, czworonożne, w kadrze portretowym), **jeleń ↔ koń** (kształt sylwetki), **samochód ↔ ciężarówka** (oba na kołach, prostokątna sylwetka). Te same pary są "trudne" dla każdego modelu — to artefakt zbioru, a nie wada twojej architektury.

### 2.5 Wizualizacja błędnych klasyfikacji

Wybierz z `test_loader` pierwszy batch, policz predykcje, znajdź indeksy obrazów, dla których predykcja **nie zgadza się** z prawdziwą etykietą. Wybierz 9 pierwszych takich obrazów i narysuj w siatce 3 × 3:

- każdy obraz w jego oryginalnej (zdenormalizowanej) wersji,
- jako tytuł: `true: {klasa_prawdziwa}, pred: {klasa_przewidziana}`.

*Denormalizacja.* Obrazy w `test_loader` przeszły przez `Normalize`, więc bezpośrednie `imshow` da dziwne kolory (wartości spoza $[0, 1]$). Aby przywrócić oryginalny zakres:

In [ ]:
#| eval: false
mean = torch.tensor(CIFAR10_MEAN).view(3, 1, 1)
std  = torch.tensor(CIFAR10_STD ).view(3, 1, 1)

def show_image(ax, img_tensor, title):
    img = (img_tensor.cpu() * std + mean).clamp(0, 1)   # (3, 32, 32) w [0, 1]
    ax.imshow(img.permute(1, 2, 0).numpy())             # (32, 32, 3) dla imshow
    ax.set_title(title, fontsize=9)
    ax.axis("off")


Obejrzyj te obrazy. Wiele będzie obiektywnie trudnych — nieostre, w nietypowym ujęciu, częściowo zasłonięte, w słabym oświetleniu. To dobry kontrapunkt dla suchej liczby "70%".

## 3. Inspekcja architektur pretrenowanych

W tej sekcji nie trenujemy niczego — otwieramy gotowe modele z `torchvision.models` i patrzymy, co jest w środku.

### 3.1 ResNet-18 — pobranie i struktura

In [ ]:
#| eval: false
model_rn18 = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
model_rn18.eval()


Argument `weights=ResNet18_Weights.IMAGENET1K_V1` pobiera wagi wytrenowane na ImageNet (1.28 mln obrazów, 1000 klas — to standardowy benchmark klasyfikacji obrazów). Bez tego argumentu (`weights=None`) dostalibyśmy tę samą architekturę z losowymi wagami — przydatne, gdy chcemy tylko policzyć parametry albo trenować od zera.

Wypisz `print(model_rn18)` i przejrzyj wyjście. Rozpoznasz znane klocki:

- `nn.Conv2d`, `nn.BatchNorm2d`, `nn.ReLU` — z sekcji 2,
- `nn.MaxPool2d`, `nn.AdaptiveAvgPool2d` — dwa rodzaje poolingu,
- `nn.Linear` — głowa klasyfikatora (`fc`).

Nowość: **`BasicBlock`**. To **blok rezydualny** (residual block), charakterystyczny dla ResNet-a. Idea: zamiast uczyć funkcję $F(x)$, uczymy "poprawkę" $F(x)$, którą **dodajemy do wejścia** — wyjście bloku to $x + F(x)$. Pozwala to trenować bardzo głębokie sieci (50+ warstw) bez problemu zanikania gradientu. Szczegóły zobaczymy na wykładzie 7; tutaj wystarczy nam, że to "blok jak każdy inny" — `nn.Module` z `forward()`.

Dostęp do konkretnych warstw:

In [ ]:
#| eval: false
print(model_rn18.conv1)        # Conv2d(3, 64, kernel_size=7, stride=2, padding=3)
print(model_rn18.fc)           # Linear(in=512, out=1000)
print(model_rn18.layer1[0])    # pierwszy BasicBlock w pierwszej grupie


ResNet-18 ma 4 grupy warstw rezydualnych (`layer1`–`layer4`), każda zawiera po dwa `BasicBlock`-i. Liczba kanałów w kolejnych grupach to 64, 128, 256, 512. Rozdzielczość przestrzenna spada w każdej grupie o połowę (przez stride=2 w pierwszej konwolucji bloku).

### 3.2 Tabela porównawcza: ile parametrów i gdzie

Napisz funkcję, która dla danego modelu zlicza:

- **całkowitą liczbę parametrów** (jedna linijka z `sum(... model.parameters())`),
- **liczbę parametrów w warstwach konwolucyjnych** (`isinstance(m, nn.Conv2d)`),
- **liczbę parametrów w warstwach liniowych** (`isinstance(m, nn.Linear)`).

Sygnatura:

In [ ]:
#| eval: false
def count_params_by_layer_type(model, layer_type):
    """Suma parametrów modułów typu `layer_type` w modelu."""
    # Wskazówka: iteruj po model.modules(); dla każdego m z isinstance(m, layer_type)
    # zsumuj p.numel() dla p w m.parameters(recurse=False) — żeby nie liczyć podrzędnych modułów dwa razy.
    ...


Wypełnij tabelę dla czterech architektur. Wagi nie zależą od inicjalizacji, więc dla ResNet-34, ResNet-50 i VGG16 użyjemy `weights=None`, żeby nie pobierać setek MB plików:

In [ ]:
#| eval: false
models = {
    "ResNet-18": model_rn18,                  # już mamy (z wagami)
    "ResNet-34": resnet34(weights=None),
    "ResNet-50": resnet50(weights=None),
    "VGG16":     vgg16(weights=None),
}

# Zbuduj DataFrame z kolumnami: Total, Conv2d, Linear, Conv/Total
...


| Model      | Parametry łącznie | Conv2d | Linear | Conv2d / łącznie |
|------------|------------------:|-------:|-------:|-----------------:|
| ResNet-18  |                   |        |        |                  |
| ResNet-34  |                   |        |        |                  |
| ResNet-50  |                   |        |        |                  |
| VGG16      |                   |        |        |                  |

**Co warto zobaczyć.**

- **ResNet-18** ma ~11,7 mln parametrów, **ResNet-34** ~21,8 mln, **ResNet-50** ~25,6 mln. Mimo że ResNet-50 jest niemal trzykrotnie głębszy niż ResNet-18, ma jedynie ~2,2 razy więcej parametrów. To zasługa bloków rezydualnych typu `Bottleneck` w ResNet-50 — używają one konwolucji 1 × 1 do redukcji wymiarowości przed kosztownymi konwolucjami 3 × 3.
- **VGG16** — ~138 mln parametrów, z czego ~123 mln (czyli ~89%) przypada na **warstwy liniowe**. Głównym winowajcą jest pierwsza warstwa klasyfikatora: `Linear(25088, 4096)` to samo w sobie ~103 mln parametrów. Konwolucje to mniej niż 11% wszystkich parametrów modelu.
- W ResNet-cie głowa klasyfikatora to pojedynczy `Linear(512, 1000)` — niespełna 0,5 mln parametrów. Cała "praca" wykonywana jest w warstwach konwolucyjnych. To projekt **głębszy i lepszy przy mniejszej liczbie parametrów** dzięki dwóm decyzjom: (1) bloki rezydualne pozwalają iść głębiej, (2) global average pooling (`AdaptiveAvgPool2d(1)`) zastępuje kosztowny `Flatten` + duża warstwa liniowa, jak miało to miejsce w VGG.
- Wniosek praktyczny dla transfer learningu: ResNet-y są mniejsze, szybsze i lepsze. To dlatego są domyślnym wyborem we współczesnych zastosowaniach, w tym w fastai.

### 3.3 Wizualizacja filtrów `conv1` w ResNet-18

Pierwsza warstwa konwolucyjna w ResNet-18 ma 64 filtry RGB o rozmiarze 7 × 7. Tensor wag ma kształt `(64, 3, 7, 7)`. Każdy filtr to mała "kratka", przy pomocy której sieć skanuje obraz wejściowy na pierwszym poziomie hierarchii cech.

In [ ]:
#| eval: false
weights = model_rn18.conv1.weight.detach().clone().cpu()   # (64, 3, 7, 7)
print(weights.shape)


Narysuj wszystkie 64 filtry w siatce 8 × 8. Każdy filtr wyświetl jako obraz RGB — normalizuj **per filtr** (min-max do $[0, 1]$):

In [ ]:
#| eval: false
def plot_filters(weights, ncols=8):
    n = weights.shape[0]
    nrows = n // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols, nrows))
    for i, ax in enumerate(axes.flat):
        f = weights[i]                                  # (3, 7, 7)
        f_min, f_max = f.min(), f.max()
        f = (f - f_min) / (f_max - f_min + 1e-8)        # do [0, 1]
        ax.imshow(f.permute(1, 2, 0).numpy())           # (7, 7, 3)
        ax.axis("off")
    plt.tight_layout()
    return fig


**Dlaczego normalizacja per filter, a nie globalnie.** Filtry sieci są niezbalansowane — niektóre mają wagi rzędu $0,5$, inne rzędu $0,05$. Gdybyśmy normalizowali wszystkie razem do wspólnego zakresu, słabsze filtry stałyby się szare i nieczytelne, a struktura ich wzorca by zniknęła. Per-filter min-max pokazuje strukturę każdego filtra niezależnie od jego skali — to standardowe podejście przy wizualizacji warstw konwolucyjnych.

Po narysowaniu obejrzyj filtry. Powinieneś rozpoznać:

- **detektory krawędzi** — pasy ciemny–jasny pod różnymi kątami,
- **detektory koloru** — filtry, w których dominuje jeden kanał RGB (czerwony, zielony, niebieski) albo prosta opozycja barw,
- **wzorce typu Gabora** — naprzemienne pasy w określonym kierunku, czułe na lokalną teksturę o określonej częstotliwości.

Te filtry to **wyuczone** detektory — nikt ich nigdy nie projektował ręcznie. Optymalizator dotarł do nich, minimalizując stratę klasyfikacji na ImageNet. Ten sam zestaw "prymitywów" (krawędzie, kolory, tekstury) wyłania się przy treningu praktycznie każdej CNN na naturalnych obrazach — to jeden z powodów, dla których pretrenowanie działa: **niskopoziomowe cechy są w dużej mierze uniwersalne** między różnymi zadaniami klasyfikacji obrazów. To intuicja, na której opiera się transfer learning z wykładu 7.

## 4. Sanity check: ResNet-18 jako klasyfikator ImageNet

Pretrenowany model w stanie surowym powinien rozpoznawać klasy, na których był trenowany. W tej krótkiej sekcji wczytamy pojedynczy obraz, przepuścimy go przez ResNet-18 i wypiszemy top-5 najbardziej prawdopodobnych klas.

### 4.1 Wczytanie i przygotowanie obrazu

Znajdź **dowolny** obraz w internecie (np. w wyszukiwarce obrazów). Najlepiej, żeby było to coś prostego, co dobrze pasuje do klas ImageNet — np. kot domowy, pies konkretnej rasy (golden retriever, labrador), słoń, samochód osobowy, łódź żaglowa. Skopiuj bezpośredni URL pliku graficznego.

Standardowy schemat — `requests` ściąga bajty, `PIL.Image.open` otwiera obraz, `transforms.Compose(...)` przekształca do postaci akceptowanej przez model:

In [ ]:
#| eval: false
URL = "..."  # wstaw bezpośredni URL obrazu (jpg/png)

response = requests.get(URL)
img = Image.open(io.BytesIO(response.content)).convert("RGB")

transform_imagenet = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std =[0.229, 0.224, 0.225]),
])

x = transform_imagenet(img).unsqueeze(0).to(device)   # (1, 3, 224, 224)

# Pokaż wczytany obraz (bez normalizacji):
fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(img); ax.axis("off")


`Resize(256)` skaluje krótszy bok do 256 pikseli, zachowując proporcje. `CenterCrop(224)` wycina kwadrat 224 × 224 ze środka. To kanoniczna procedura preprocessingu dla wszystkich modeli ImageNet z `torchvision`.

**Dlaczego dokładnie te statystyki normalizacji.** Wartości `mean = [0.485, 0.456, 0.406]`, `std = [0.229, 0.224, 0.225]` to średnia i odchylenie standardowe per kanał obliczone na **całym zbiorze treningowym ImageNet**. Model był trenowany na obrazach po takiej normalizacji — gdybyśmy wstawili obraz w innej skali (np. surowe wartości $[0, 1]$ bez odejmowania średniej), pierwsza warstwa konwolucyjna otrzymałaby dane statystycznie obce od tych, na których uczyła się rozpoznawać krawędzie i kolory, i działałaby gorzej. To ten sam powód, dla którego scaler dopasowywaliśmy na zbiorze treningowym i stosowaliśmy do testu w lab 5: model musi widzieć dane w tej samej dystrybucji, na której był uczony.

### 4.2 Forward pass i top-5 predykcji

In [ ]:
#| eval: false

model_rn18 = model_rn18.to(device)

model_rn18.eval()
with torch.no_grad():
    logits = model_rn18(x)              # (1, 1000)
    probs  = F.softmax(logits, dim=1)
    topk_probs, topk_idx = probs[0].topk(5)


Aby zamienić indeksy 0–999 na nazwy klas, pobierz listę klas ImageNet (1000 nazw, jedna na wiersz):

In [ ]:
#| eval: false
CLASSES_URL = "https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt"
imagenet_classes = requests.get(CLASSES_URL).text.strip().split("\n")
assert len(imagenet_classes) == 1000

for p, i in zip(topk_probs.cpu(), topk_idx.cpu()):
    print(f"{p.item()*100:5.1f}%  {imagenet_classes[i]}")


Dla zdjęcia kota powinieneś zobaczyć w top-1 jedną z klas typu *tabby*, *Egyptian cat*, *tiger cat* — ImageNet ma kilkanaście odmian kotów. Pies → *golden retriever*, *Labrador retriever*, *Chesapeake Bay retriever* itp. Jeśli top-1 brzmi sensownie, sanity check jest zaliczony — pretrenowany model w stanie surowym faktycznie klasyfikuje obrazy.

*Co jeśli wynik wygląda dziwnie.* Najczęstsza przyczyna: **pominięta normalizacja**. Bez `Normalize(...)` wartości pikseli mieszczą się w $[0, 1]$, model dostaje obraz "za jasny" o ~0,45 na każdym kanale i top-1 dryfuje na losowe klasy. Druga przyczyna: obraz wczytany jako jednokanałowy (paleta lub grayscale) — pamiętaj o `.convert("RGB")` po `Image.open`, bo `Conv2d(3, ...)` oczekuje trzech kanałów.

## 5. Zadania dodatkowe

### 5.1 Lepsze accuracy na CIFAR-10

Rozszerz architekturę z sekcji 2 (więcej kanałów, czwarty blok, większa głowa MLP) lub zmień hiperparametry treningu (więcej epok, lr-scheduler typu `optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)` wywoływany po każdej epoce, weight decay w optymalizatorze przez `optim.Adam(..., weight_decay=1e-4)`). Spróbuj osiągnąć ≥80% accuracy testowej. Zanotuj zmianę i jej wpływ.

### 5.2 Global Average Pooling zamiast Flatten

W standardowych architekturach z ostatniej dekady (ResNet, MobileNet, EfficientNet) głowa klasyfikatora to **Global Average Pooling** — uśrednienie każdej mapy cech do skalara — zamiast `Flatten` + duża warstwa `Linear`. To drastycznie redukuje liczbę parametrów. Zaimplementuj wariant CNN z sekcji 2, w którym ostatni MaxPool jest zastąpiony przez `nn.AdaptiveAvgPool2d(1)`, a klasyfikator to po prostu `Linear(128, 10)`. Policz różnicę w liczbie parametrów (powinno być rzędu kilkudziesięciu razy mniej). Porównaj accuracy z modelem podstawowym.

### 5.3 Augmentation off

Wytrenuj model z sekcji 2 **bez** `RandomCrop` i `RandomHorizontalFlip` (zostaw w `transform_train` tylko `ToTensor` + `Normalize`). Co dzieje się z krzywymi treningu i testu? Spodziewaj się znacznie większego rozjazdu między train acc i test acc — to klasyczny overfitting. Po ilu epokach jest on widoczny? Czy train accuracy dochodzi do 100%?

### 5.4 Receptive field ResNet-18

Stosując wzór z sekcji 1.2, policz receptive field na wyjściu pierwszego `BasicBlock`-u w `layer1` ResNet-18. Wskazówka: zanim dojdziesz do `layer1`, sieć ma `conv1` (k=7, s=2, p=3) i `nn.MaxPool2d(kernel_size=3, stride=2, padding=1)` — odczytaj to z `print(model_rn18)`. Pamiętaj, że `BasicBlock` w `layer1` to dwie konwolucje 3 × 3 (z padding=1, stride=1). Ile pikseli oryginalnego obrazu 224 × 224 "widzi" pojedynczy neuron na wyjściu `layer1`?

### 5.5 Trudniejszy obraz dla sanity check

Powtórz sanity check z sekcji 4 dla obrazu, który **nie jest** oczywistym przedstawicielem żadnej z klas ImageNet — np. zdjęcie portretowe człowieka (ImageNet nie ma klasy "człowiek" jako takiej, tylko np. *scuba diver* albo *groom*), abstrakcyjną grafikę, albo zdjęcie typowo polskiego krajobrazu. Co model "widzi"? Jak rozkłada się prawdopodobieństwo na top-5? Czy któraś klasa wygrywa z dużą pewnością, czy rozkład jest płaski?